# 05 – Feature Fusion (Tabular + CNN Embeddings)

**Project:** Multi-Agent DRL + CNN Alternative Data for Credit Decisioning in Emerging Markets

This notebook loads the trained CNN encoder, extracts embeddings from transaction sequences, concatenates them with tabular features, and saves a fused dataset ready for DRL agents (supports RQ2).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "german_credit_model.csv")
txn_seq = np.load(DATA_PROCESSED / "transaction_sequences.npy")

X_tab = df.drop(columns=['default']).values.astype(np.float32)
y = df['default'].values
thin = df['thin_file_flag'].values

print(f"Tabular shape: {X_tab.shape}")
print(f"Sequence shape: {txn_seq.shape}")

In [ ]:
class TransactionCNNEncoder(nn.Module):
    def __init__(self, in_channels=8, seq_len=30, embedding_dim=32):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(16)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(32)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, embedding_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

encoder = TransactionCNNEncoder(embedding_dim=32).to(device)
encoder_path = RESULTS / "cnn_encoder.pt"
if encoder_path.exists():
    encoder.load_state_dict(torch.load(encoder_path, map_location=device))
    print("CNN Encoder loaded successfully.")
else:
    print("Warning: cnn_encoder.pt not found. Using randomly initialized encoder.")
encoder.eval()

In [ ]:
with torch.no_grad():
    seq_tensor = torch.tensor(txn_seq, dtype=torch.float32).to(device)
    embeddings = encoder(seq_tensor).cpu().numpy()

print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
scaler = StandardScaler()
X_tab_scaled = scaler.fit_transform(X_tab)

X_fused = np.concatenate([X_tab_scaled, embeddings], axis=1)
print(f"Fused feature shape: {X_fused.shape}")

np.save(DATA_PROCESSED / "X_fused.npy", X_fused)
np.save(DATA_PROCESSED / "y.npy", y)
np.save(DATA_PROCESSED / "thin.npy", thin)

fused_df = pd.DataFrame(X_fused)
fused_df['default'] = y
fused_df['thin_file_flag'] = thin
fused_df.to_csv(DATA_PROCESSED / "fused_features.csv", index=False)

print("\nSaved files:")
print("  - data/processed/X_fused.npy")
print("  - data/processed/y.npy")
print("  - data/processed/thin.npy")
print("  - data/processed/fused_features.csv")
print("\nFeature Fusion complete. Ready for DRL agents.")